In [16]:
using ChainRulesCore
using FiniteDifferences
using PyCall
using Zygote
using MarineHydro

cpt = pyimport("capytaine")
mesh_module = pyimport("meshio._mesh")
mesh = mesh_module.Mesh
gmsh = pyimport("gmsh")
pygmsh = pyimport("pygmsh")
meshio_module = pyimport("capytaine.io.meshio")
load_from_meshio = meshio_module.load_from_meshio




PyObject <function load_from_meshio at 0x000001D0BF5FFB00>

In [17]:
# AquaHarmonics hull
# https://github.com/sandialabs/WecOptTool/blob/main/wecopttool/geom.py
function cptMeshAH(r1)

        T1 = 1.5
        T2 = 0.355
        T3 = 7.25
        # r1 = 1.085
        r2 = 0.405
        r3 = 0.355

        mesh_size_factor = 0.8
        ofst = 0
        geom = pygmsh.occ.Geometry() 
        gmsh.initialize()
        gmsh.option.setNumber("Mesh.MeshSizeFactor", mesh_size_factor)
        cyl1 = geom.add_cylinder([0, 0, 0],
                                [0, 0, -T1],
                                r1)

        cone = geom.add_cone([0, 0, -T1],
                [0, 0, -T2],
                r1,
                r2)
        cylout = geom.add_cylinder([0, 0, -1*(T1+T2)],
                                [0, 0, -T3],
                                r2)
        cylin = geom.add_cylinder([0, 0, -1*(T1+T2)],
                                [0, 0, -T3],
                                r3)
        cyl2 = geom.boolean_difference(cylout, cylin)[1]
        wecGeom = geom.boolean_union(entities=[cyl1, cone, cyl2],
                                        delete_first=true)[1]

        geom.translate(wecGeom, [0, 0, ofst])
        mesh = geom.generate_mesh()
        cptmesh = load_from_meshio(mesh, "AquaHarmonics")
        return cptmesh.keep_immersed_part(inplace=true)
end

       

cptMeshAH (generic function with 1 method)

In [18]:
T1 = 1.5
T2 = 0.355
T3 = 7.25
r1 = 1.085
r2 = 0.405
r3 = 0.355

cptahmesh = cptMeshAH(r1)

PyObject Mesh(vertices=[[... 163 vertices ...]], faces=[[... 315 faces ...]], name="AquaHarmonics")

In [19]:
# Functions for generating input to MarineHydro Mesh struct 
function get_mesh_sizes(r1) 
    cptmesh = cptMeshAH(r1)
    return cptmesh.nb_faces, cptmesh.nb_vertices
end
function get_vertices(r1) 
    cptmesh = cptMeshAH(r1)
    return reduce(vcat,cptmesh.vertices)
end
function get_centers(r1) 
    cptmesh = cptMeshAH(r1)
    return reduce(vcat,cptmesh.faces_centers)
end
function get_normals(r1) 
    cptmesh = cptMeshAH(r1)
    return reduce(vcat,cptmesh.faces_normals)
end
function get_radii(r1) 
    cptmesh = cptMeshAH(r1)
    return reduce(vcat,cptmesh.faces_radiuses)
end
function get_areas(r1) 
    cptmesh = cptMeshAH(r1)
    return reduce(vcat,cptmesh.faces_areas)
end
function get_faces(r1) 
    cptmesh = cptMeshAH(r1)
    return reduce(vcat,cptmesh.faces)
end

# Functions for chainrules
h = 1e-3
∂J_r_fd(f,r) = (f(r+h) .- f(r-h)) ./ (2*h)
∂J_r_fd(f,r; h=1e-5) = (f(r+h) .- f(r-h)) ./ (2*h)

function ChainRulesCore.rrule(::typeof(get_vertices), r)
    y = get_vertices(r)
    function f_pullback(dy)
        df = NoTangent()  # No gradient w.r.t the function
        dv = ∂J_r_fd(get_vertices,r)  # Finite difference derivative with respect to `r`
        dx = dv' * dy    #pullback
        return (df, dx)  
    end    
    return y, f_pullback
end

function ChainRulesCore.rrule(::typeof(get_centers), r)
    y = get_centers(r)
    function f_pullback(dy)
        df = NoTangent()  # No gradient w.r.t the function
        dv = ∂J_r_fd(get_centers, r)  # Finite difference derivative with respect to `r`
        dx = dv' * dy    #pullback
        return (df, dx)  
    end
    return y, f_pullback
 end

function ChainRulesCore.rrule(::typeof(get_normals), r)
    y = get_normals(r)
    function f_pullback(dy)
        df = NoTangent()  # No gradient w.r.t the function
        dv = ∂J_r_fd(get_normals, r)  # Finite difference derivative with respect to `r`
        dx = dv' * dy    #pullback
        return (df, dx)  
    end
    return y, f_pullback
  end

function ChainRulesCore.rrule(::typeof(get_radii), r)
    y = get_radii(r)
    function f_pullback(dy)
        df = NoTangent()  # No gradient w.r.t the function
        dv = ∂J_r_fd(get_radii, r)  # Finite difference derivative with respect to `r`
        dx = dv' * dy    #pullback
        return (df, dx)  
    end
    return y, f_pullback
end

function ChainRulesCore.rrule(::typeof(get_areas), r)
    y = get_areas(r)
    function f_pullback(dy)
        df = NoTangent()  # No gradient w.r.t the function
        dv = ∂J_r_fd(get_areas, r)  # Finite difference derivative with respect to `r`
        dx = dv' * dy    #pullback
        return (df, dx)  
    end
    return y, f_pullback
  end

  function ChainRulesCore.rrule(::typeof(get_faces), r)
    y = get_faces(r)
    function f_pullback(dy)
        df = NoTangent()  # No gradient w.r.t the function
        dv = ∂J_r_fd(get_faces, r)  # Finite difference derivative with respect to `r`
        dx = dv' * dy    #pullback
        return (df, dx)  
    end
    return y, f_pullback
  end





function differentiableMesh(r1)
    nf,nv = Zygote.@ignore get_mesh_sizes(r1) #hack..need to know how many first to reshape
    vertices = reshape(get_vertices(r1),(nv,3))
    faces =  reshape(get_faces(r1),(nf,4))
    centers = reshape(get_centers(r1),(nf,3))
    normals = reshape(get_normals(r1),(nf,3))
    areas = get_areas(r1)
    radii = get_radii(r1)
    nvertices = size(vertices,1)
    nfaces = size(centers,1)
    return Mesh(vertices,faces,centers,normals,areas,radii,nvertices,nfaces)
end






differentiableMesh (generic function with 1 method)

In [20]:

MHmesh = differentiableMesh(r1)
omega = 0.3
dof = [0,0,1]
AddedMass = calculate_radiation_forces(MHmesh,dof,omega)[1]

2308.706416965309

In [21]:
omega = 1.1
dof = [0.0,0.0,1.0]
function added_mass_program(radius,omega,dof)  
    mesh = differentiableMesh(radius) #fd
    A = calculate_radiation_forces(mesh,dof,omega)[1]
    return A
end

A(radius) = added_mass_program(radius,omega,dof)  

A (generic function with 1 method)

In [22]:
sum(MHmesh.vertices[:,3])

-723.6725277561314

In [23]:
r = 1
differentiableMesh(r1)

function fun(r1)
    MHmesh = differentiableMesh(r1)
    return sum(MHmesh.vertices[:,1])
end
    
grad, = Zygote.gradient(fun,r)

(0.0,)

In [24]:
fun(3)

3.991571018012395

In [ ]:
A_r_grad1, = Zygote.gradient(A,r)